
# **Step 1:** The Ingestion Block (Connecting & Extracting)

This block sets up a secure connection socket to your locally running MetaTrader 5 terminal, requests a clean slice of historical time-series data, and converts the raw binary structure into a localized Pandas DataFrame.
Create a new cell and write the following code:

In [17]:
import torch  
import numpy as np
import pandas as pd
import torch.nn as nn
import MetaTrader5 as mt5 
import torch.optim as optim 
from xgboost import XGBClassifier
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, accuracy_score 
from sklearn.linear_model import LogisticRegression

print("Dependencies Downloaded Successfully")

Dependencies Downloaded Successfully


In [18]:
def connect_and_fetch_forex_data(symbol:str, timeframe,num_bars:int) -> pd.DataFrame: 
    if not mt5. initialize():
        raise RuntimeError(f"Could not Connect to MT5. \n Error: {mt5.last_error()}")

    print("Initialized Connection to MT5 Terminal.....")

    mt5.symbol_select(symbol,True)

    raw_rates = mt5.copy_rates_from_pos(symbol,timeframe,0,num_bars)

    mt5.shutdown()
    print("MT5 Terminal Closed Successfully!")

    df = pd.DataFrame(raw_rates)

    df['time'] = pd.to_datetime(df['time'],unit='s')
    df.set_index('time',inplace=True)

    essential_columns = ['open', 'high', 'low', 'close', 'tick_volume']
    return df[essential_columns]

raw_eurusd_df =  connect_and_fetch_forex_data("EURUSD",mt5.TIMEFRAME_H1,30000)
print(f"Matrix Shape Extracted: {raw_eurusd_df.shape}")

Initialized Connection to MT5 Terminal.....
MT5 Terminal Closed Successfully!
Matrix Shape Extracted: (30000, 5)


# **Step 2 :** Feature Enginering, here I am gonna take the raw rates and then create the Engineered stationary features.

In [19]:
def engineer_stationary_features(raw_df: pd.DataFrame) -> pd.DataFrame: 
    matrix = raw_df.copy()

    matrix['log_return'] = np.log(matrix['close']/matrix['close'].shift(1))

    matrix['high_low_span'] = matrix['high'] - matrix['low']
    matrix['volatility_atr'] = matrix['high_low_span'].rolling(window=14).mean()

    matrix['raw_hour'] = matrix.index.hour
    
    matrix['hour_sin'] = np.sin(2 * np.pi * matrix['raw_hour'] / 24.0)
    matrix['hour_cos'] = np.cos(2 * np.pi * matrix['raw_hour'] / 24.0)

    matrix.dropna(inplace=True)

    final_features = matrix.drop(columns=['open','high','low','raw_hour','high_low_span'])

    return final_features 

final_feature_matrix = engineer_stationary_features(raw_eurusd_df)
print("Pipeline complete! Displaying top 3 rows of processed ML input matrix:")
print(final_feature_matrix[['close', 'log_return', 'volatility_atr', 'hour_sin', 'hour_cos']].head(3))

Pipeline complete! Displaying top 3 rows of processed ML input matrix:
                       close  log_return  volatility_atr  hour_sin  hour_cos
time                                                                        
2021-12-14 20:00:00  1.12650   -0.000612        0.001359 -0.866025  0.500000
2021-12-14 21:00:00  1.12560   -0.000799        0.001389 -0.707107  0.707107
2021-12-14 22:00:00  1.12541   -0.000169        0.001374 -0.500000  0.866025


 This function contains the loop that steps through your historical rows, projects the dynamic ATR brackets forward in time, and extracts the target ground-truth label.

In [20]:
def generate_triple_barrier_labels(df: pd.DataFrame, pt_mult: float = 2.0, sl_mult: float = 1.0, max_bars: int = 24) -> pd.Series:

    labels = np.zeros(len(df))
    
    close_prices = df['close'].values
    atr_values = df['volatility_atr'].values
    
    total_rows = len(df)
    
    for t in range(total_rows - max_bars):
        current_close = close_prices[t]
        current_atr = atr_values[t]
        
        upper_barrier = current_close + (current_atr * pt_mult)
        lower_barrier = current_close - (current_atr * sl_mult)
        
        forward_path = close_prices[t + 1 : t + 1 + max_bars]
        
        profit_crossings = np.where(forward_path >= upper_barrier)[0]
        loss_crossings = np.where(forward_path <= lower_barrier)[0]
        
        first_profit_idx = profit_crossings[0] if len(profit_crossings) > 0 else None
        first_loss_idx = loss_crossings[0] if len(loss_crossings) > 0 else None
        
        if first_profit_idx is not None and first_loss_idx is not None:
            if first_profit_idx < first_loss_idx:
                labels[t] = 1  
            else:
                labels[t] = -1  
        elif first_profit_idx is not None:
            labels[t] = 1      
        elif first_loss_idx is not None:
            labels[t] = -1      
        else:
            labels[t] = 0       
            
    return pd.Series(labels, index=df.index, name='target')

target_labels = generate_triple_barrier_labels(final_feature_matrix, pt_mult=2.0, sl_mult=1.0, max_bars=24)

final_feature_matrix['target'] = target_labels
clean_ml_matrix = final_feature_matrix.iloc[:-24].copy()

print(f"Labeling complete! Processed Dataset Shape: {clean_ml_matrix.shape}")


Labeling complete! Processed Dataset Shape: (29963, 7)


In [21]:
class_counts = clean_ml_matrix['target'].value_counts()
print("--- TARGET DISTRIBUTION BALANCE ---")
print(f"Take Profit (+1.0): {class_counts.get(1.0, 0)} trades")
print(f"Stop Loss (-1.0):   {class_counts.get(-1.0, 0)} trades")
print(f"Time Expiry (0.0):  {class_counts.get(0.0, 0)} trades\n")

class_percentages = clean_ml_matrix['target'].value_counts(normalize=True) * 100
print("--- PERCENTAGE PROFILE ---")
print(f"Take Profit: {class_percentages.get(1.0, 0):.2f}%")
print(f"Stop Loss:   {class_percentages.get(-1.0, 0):.2f}%")
print(f"Time Expiry: {class_percentages.get(0.0, 0):.2f}%")

assert class_counts.get(0.0, 0) < (len(clean_ml_matrix) * 0.85), "Warning: Over 85% of your data is marked as Time Expiry. Your ATR barriers may be too wide!"

assert set(clean_ml_matrix['target'].unique()).issubset({-1.0, 0.0, 1.0}), "Error: Invalid classes discovered in target array!"

print("Sprint 2 Data Target Assembly complete! Ready for Purged Cross-Validation.")



--- TARGET DISTRIBUTION BALANCE ---
Take Profit (+1.0): 9820 trades
Stop Loss (-1.0):   18104 trades
Time Expiry (0.0):  2039 trades

--- PERCENTAGE PROFILE ---
Take Profit: 32.77%
Stop Loss:   60.42%
Time Expiry: 6.81%
Sprint 2 Data Target Assembly complete! Ready for Purged Cross-Validation.


We will take the first 80% of our timeline for training and reserve the final 20% for testing, adding a 24-hour purge wall between them.

In [22]:
def create_purged_train_test_split(df: pd.DataFrame, target_col: str, max_bars: int = 24, train_ratio: float = 0.8):
    total_rows = len(df)

    split_index = int(total_rows * train_ratio)

    train_df = df.iloc[:split_index].copy()
    test_df = df.iloc[split_index + max_bars:].copy()

    feature_cols = ['log_return','volatility_atr','hour_cos','hour_sin',]

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    
    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    print("--- PURGED SPLIT VERIFICATION ---")
    print(f"Training Features Shape (X_train): {X_train.shape}")
    print(f"Testing Features Shape  (X_test):  {X_test.shape}")
    print(f"Data Points Purged in Buffer Gap:   {max_bars} bars\n")

    return X_train,y_train,X_test,y_test 

    
X_train, y_train, X_test,y_test = create_purged_train_test_split(clean_ml_matrix,target_col='target')


--- PURGED SPLIT VERIFICATION ---
Training Features Shape (X_train): (23970, 4)
Testing Features Shape  (X_test):  (5969, 4)
Data Points Purged in Buffer Gap:   24 bars



This cell is to handle the Imbalance that arose when we ivestigates in the 2nd cell

In [23]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

print("Sample weights generated successfully!")
print(f"First 5 row penalization values: {sample_weights_train[:5]}")


Sample weights generated successfully!
First 5 row penalization values: [4.88685015 4.88685015 0.99464708 0.99464708 0.99464708]


Going to initilize my first basse classifications model. Deliverables: I have finished handling and peforming feature Engineering to the data now lets go to the first base model.

In [24]:
y_train_mapped = y_train.map({-1.0:0,0.0:1,1.0:2})
y_test_mapped = y_test.map({-1.0:0,0.0:1,1.0:2})

xgb_analyst = XGBClassifier(
    n_estimators=150,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    random_state=42,
    tree_method='hist' 
)

print("Training XGBoost on our sequential data split....")
# Fitting the Training data to the XgBoost 

xgb_analyst.fit(
    X_train,
    y_train_mapped,
    sample_weight=sample_weights_train
)

print("**DONE** XGBOOST Training Complete")

Training XGBoost on our sequential data split....


**DONE** XGBOOST Training Complete


Going to test the Trained model on the unseen data that I splitted in the above train_test_split cell.

In [25]:
y_pred = xgb_analyst.predict(X_test)

print("====XGBOOST PEFORMANCE REPORT====")
print(f"Overall Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test_mapped, y_pred, target_names=['Stop Loss (-1)', 'Time Expiry (0)', 'Take Profit (1)']))

====XGBOOST PEFORMANCE REPORT====
Overall Test Accuracy: 7.72%

                 precision    recall  f1-score   support

 Stop Loss (-1)       0.64      0.30      0.41      3782
Time Expiry (0)       0.14      0.54      0.22       404
Take Profit (1)       0.33      0.47      0.39      1783

       accuracy                           0.37      5969
      macro avg       0.37      0.44      0.34      5969
   weighted avg       0.51      0.37      0.39      5969



Going to create the second classifier (the Long Short Term Memory -LSTM-) Neural Network. But at first since the LSTM NNs expect a 3D Tensor (Samples, Time-stamps, Features) I need to first Engineer it before.

In [26]:
def create_3d_lstm_sequence(X_data:pd.DataFrame, y_data: pd.Series, lookback_window: int = 24):
    X_3d,y_3d = [],[]

    X_values = X_data.values
    y_values = y_data.values

    for i in range(lookback_window, len(X_data)):
        X_3d.append(X_values[i - lookback_window : i])
        y_3d.append(y_values[i])
        
    return np.array(X_3d), np.array(y_3d)

y_train_mapped = y_train.map({-1.0:0,0.0:1,1.0:2})
y_test_mapped = y_test.map({-1.0:0,0.0:1,1.0:2})

x_train_3d, y_train_3d = create_3d_lstm_sequence(X_train,y_train_mapped,lookback_window=24)
x_test_3d, y_test_3d = create_3d_lstm_sequence(X_test,y_test_mapped,lookback_window=24)

print("=== DEEP LEARNING TENSOR VERIFICATION ===")
print(f" X_train_3d Shape: {x_train_3d.shape} -> [Samples, Hours, Features]")
print(f" y_train_3d Shape: {y_train_3d.shape}")


=== DEEP LEARNING TENSOR VERIFICATION ===
 X_train_3d Shape: (23946, 24, 4) -> [Samples, Hours, Features]
 y_train_3d Shape: (23946,)


In [27]:
torch.manual_seed(42)

class ForexLSTMAnalyst(nn.Module):
    def __init__(self,input_dim:int,hidden_dim:int,output_dim:int=3,num_layers:int=3):
        super(ForexLSTMAnalyst,self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0.0 
        )
        self.fc = nn.Linear(hidden_dim,output_dim)

    def forward(self,x):
        h0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(x.device)
        c0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(x.device)

        out,_ = self.lstm(x,(h0,c0))

        final_time_step_output = out[:, -1, :]

        return self.fc(final_time_step_output)

lstm_analyst = ForexLSTMAnalyst(input_dim=4, hidden_dim=64, num_layers=2)
print("PyTorch LSTM Architecture compiled successfully.")
print(lstm_analyst)

PyTorch LSTM Architecture compiled successfully.
ForexLSTMAnalyst(
  (lstm): LSTM(4, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=3, bias=True)
)


In [28]:
x_train_tensor = torch.tensor(x_train_3d,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_3d,dtype=torch.long)

x_test_tensor = torch.tensor(x_test_3d,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_3d,dtype=torch.long)

train_dataset = TensorDataset(x_train_tensor,y_train_tensor)
test_dataset = TensorDataset(x_test_tensor,y_test_tensor)

train_loader = DataLoader(train_dataset,batch_size=64,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)

print(f"Pipeline Established. Batch Training contains {len(train_loader)} iteration Circles")

Pipeline Established. Batch Training contains 375 iteration Circles


# Now the following cell is to be used for implementing 
# CrossEntropyLoss as our objective optimizer (which automatically handles multi-class targets) and the **Adam Optimizer**.

In [29]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_analyst.parameters(), lr=0.001)

epochs = 15 
print("Initializing LSTM Training Loop accross Sequential Batches....")

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        outputs = lstm_analyst(batch_X)
        
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        
        optimizer.step()
        
        epoch_loss += loss.item() * batch_X.size(0)
        
    total_epoch_loss = epoch_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_epoch_loss:.4f}")

print("LSTM Network weights compiled and locked down!")

Initializing LSTM Training Loop accross Sequential Batches....
Epoch [1/15] - Loss: 0.8645
Epoch [2/15] - Loss: 0.8503
Epoch [3/15] - Loss: 0.8493
Epoch [4/15] - Loss: 0.8488
Epoch [5/15] - Loss: 0.8485
Epoch [6/15] - Loss: 0.8485
Epoch [7/15] - Loss: 0.8482
Epoch [8/15] - Loss: 0.8481
Epoch [9/15] - Loss: 0.8480
Epoch [10/15] - Loss: 0.8479
Epoch [11/15] - Loss: 0.8479
Epoch [12/15] - Loss: 0.8477
Epoch [13/15] - Loss: 0.8476
Epoch [14/15] - Loss: 0.8477
Epoch [15/15] - Loss: 0.8476
LSTM Network weights compiled and locked down!


In [30]:
lstm_analyst.eval()
all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = lstm_analyst(batch_X)
        _, predicted_classes = torch.max(outputs, 1)
        
        all_predictions.extend(predicted_classes.numpy())
        all_true_labels.extend(batch_y.numpy())

print("==== LSTM PERFORMANCE REPORT ====")
print(f"Overall Test Accuracy: {accuracy_score(all_true_labels, all_predictions) * 100:.2f}%\n")
print(classification_report(all_true_labels, all_predictions, target_names=['Stop Loss [0]', 'Time Expiry [1]', 'Take Profit [2]']))


==== LSTM PERFORMANCE REPORT ====
Overall Test Accuracy: 63.40%

                 precision    recall  f1-score   support

  Stop Loss [0]       0.63      1.00      0.78      3769
Time Expiry [1]       0.00      0.00      0.00       404
Take Profit [2]       0.00      0.00      0.00      1772

       accuracy                           0.63      5945
      macro avg       0.21      0.33      0.26      5945
   weighted avg       0.40      0.63      0.49      5945



c:\Users\HP\anaconda3\envs\void\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\anaconda3\envs\void\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\HP\anaconda3\envs\void\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [31]:
xgb_train_probs = xgb_analyst.predict_proba(X_train)
xgb_test_probs = xgb_analyst.predict_proba(X_test)

lstm_analyst.eval()
lstm_train_probs=[]
lstm_test_probs=[]

softmax = torch.nn.Softmax(dim=1)

with torch.no_grad():
    for batch_X, _ in train_loader:
        logits = lstm_analyst(batch_X)
        probs = softmax(logits)
        lstm_train_probs.extend(probs.numpy())
        
    for batch_X, _ in test_loader:
        logits = lstm_analyst(batch_X)
        probs = softmax(logits)
        lstm_test_probs.extend(probs.numpy())

lstm_train_probs = np.array(lstm_train_probs)
lstm_test_probs = np.array(lstm_test_probs)

xgb_train_probs_aligned = xgb_train_probs[24:]
xgb_test_probs_aligned = xgb_test_probs[24:]
y_train_aligned = y_train_mapped.values[24:]
y_test_aligned = y_test_mapped.values[24:]

X_blend_train = np.hstack((xgb_train_probs_aligned, lstm_train_probs))
X_blend_test = np.hstack((xgb_test_probs_aligned, lstm_test_probs))

print("=== BOARDROOM MATRIX STACK COMPLETE ===")
print(f" Blend Train Features Matrix Shape: {X_blend_train.shape}")
print(f" Blend Test Features Matrix Shape:  {X_blend_test.shape}")

=== BOARDROOM MATRIX STACK COMPLETE ===
 Blend Train Features Matrix Shape: (23946, 6)
 Blend Test Features Matrix Shape:  (5945, 6)


In [32]:
meta_manager = LogisticRegression(class_weight='balanced', random_state=42)

print("Optimizing Meta-Manager voting weights...")
meta_manager.fit(X_blend_train, y_train_aligned)

final_predictions = meta_manager.predict(X_blend_test)

print("==== FINAL ENSEMBLE SYSTEM REPORT ====")
print(f"Overall Balanced Accuracy: {accuracy_score(y_test_aligned, final_predictions) * 100:.2f}%\n")
print(classification_report(y_test_aligned, final_predictions, target_names=['Stop Loss (-1)', 'Time Expiry (0)', 'Take Profit (1)']))


Optimizing Meta-Manager voting weights...
==== FINAL ENSEMBLE SYSTEM REPORT ====
Overall Balanced Accuracy: 41.16%

                 precision    recall  f1-score   support

 Stop Loss (-1)       0.63      0.41      0.50      3769
Time Expiry (0)       0.14      0.41      0.21       404
Take Profit (1)       0.32      0.42      0.36      1772

       accuracy                           0.41      5945
      macro avg       0.36      0.41      0.36      5945
   weighted avg       0.50      0.41      0.44      5945

